# 面试问题：LLM 预训练数据配比怎样设计？比例采样、temperature sampling 与 DoReMi 思路是什么？

**一句话回答**：先把每个 domain 的许可、质量、唯一 token 和目标用途做成版本化合同；原始 token 比例会让大 Web 域淹没小而重要的代码/数学域，temperature sampling 用 `q_i∝n_i^α` 平滑规模，DoReMi 类方法再依据 proxy 模型相对 reference 的域损失动态调权。最终权重仍需上下限、去重归属、预算和 held-out 多域评测约束。

本 Notebook 从零实现域合同、温度配比、整数 token quota、可复现采样、简化 Group-DRO 更新、上下限投影、跨域重复归属和 mixture drift 监控。代码注释解释关键公式和边界。


In [ ]:
from dataclasses import dataclass  # 导入本单元所需的依赖。
import hashlib,math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

# 固定随机种子，保证采样和测试可复现。
SEED139=13901; rng139=np.random.default_rng(SEED139)  # 计算并保存当前步骤的中间状态。
assert SEED139==13901  # 用受控断言验证关键不变量。
assert math.isclose(sum([.5,.3,.2]),1.0)  # 用受控断言验证关键不变量。
assert np.isfinite(rng139.normal())  # 用受控断言验证关键不变量。


## 1. Domain 不是一个随意文件夹名

每域记录快照、语言、来源、许可证、唯一 token、质量门禁和 PII/删除策略。`tokens` 应是最终 tokenizer 与过滤版本下的数量，而不是下载字节。许可不允许的域即使 loss 很高也不能被优化器加权。


In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Domain139:  # 定义承载本节状态与行为的数据结构。
    name:str; tokens:int; quality:float; allowed:bool; snapshot:str  # 执行当前语句以推进本节示例。
    def __post_init__(self):  # 定义本节可复用的核心函数。
        # 非法规模或质量在进入混合器前失败关闭。
        if self.tokens<=0 or not 0<=self.quality<=1: raise ValueError("domain_contract")  # 按当前条件选择后续控制路径。
domains139=[Domain139("web",1_000_000,.72,True,"s1"),Domain139("code",100_000,.9,True,"s1"),Domain139("math",10_000,.95,True,"s1")]  # 计算并保存当前步骤的中间状态。
assert len({d.name for d in domains139})==3  # 用受控断言验证关键不变量。
assert all(d.allowed for d in domains139)  # 用受控断言验证关键不变量。
try: Domain139("bad",0,.5,True,"s1"); raise AssertionError("bad domain")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="domain_contract"  # 捕获预期异常并验证失败分支。


## 2. Temperature sampling 平滑大域优势

令原始规模为 `n_i`，采样概率 `q_i=n_i^α/Σn_j^α`。`α=1` 等于规模比例，`α=0` 等于各域均匀；越小越提升低资源域，但也会增加重复 epoch 和过拟合风险。α 必须与总训练 token 一起报告。


In [ ]:
def temperature_probs139(sizes,alpha):  # 定义本节可复用的核心函数。
    # 对规模取幂后归一化，不把 temperature 与 softmax 温度混淆。
    x=np.asarray(sizes,dtype=float)**alpha; return x/x.sum()  # 计算并保存当前步骤的中间状态。
sizes139=[d.tokens for d in domains139]; p1_139=temperature_probs139(sizes139,1); p05_139=temperature_probs139(sizes139,.5)  # 计算并保存当前步骤的中间状态。
assert np.allclose(p1_139,np.array(sizes139)/sum(sizes139))  # 用受控断言验证关键不变量。
assert p05_139[-1]>p1_139[-1]  # 用受控断言验证关键不变量。
assert math.isclose(float(p05_139.sum()),1.0)  # 用受控断言验证关键不变量。


## 3. 概率要落成精确整数 token quota

训练计划最终需要整数 token/sequence。先取 floor，再按小数余量最大的域分配剩余 token，可保证总量精确且单域误差小于 1。分布式 worker 应从同一个全局 quota/epoch manifest 切 shard，不能各自独立四舍五入。


In [ ]:
def largest_remainder139(probs,total):  # 定义本节可复用的核心函数。
    raw=np.asarray(probs)*total; q=np.floor(raw).astype(int)  # 计算并保存当前步骤的中间状态。
    # 剩余名额按小数部分从大到小分配。
    order=np.argsort(-(raw-q),kind="stable")  # 计算并保存当前步骤的中间状态。
    for i in order[:total-q.sum()]: q[i]+=1  # 遍历输入元素以累积或检查结果。
    return q  # 返回当前分支计算出的结果。
quota139=largest_remainder139(p05_139,101)  # 计算并保存当前步骤的中间状态。
assert quota139.sum()==101  # 用受控断言验证关键不变量。
assert np.max(np.abs(quota139-101*p05_139))<1  # 用受控断言验证关键不变量。
assert all(quota139>0)  # 用受控断言验证关键不变量。


## 4. 采样器保存 RNG 与消费进度

用累计分布把随机数映射到域，域内再由可恢复 sampler 选择文档。checkpoint 必须包含 mixture revision、每域 cursor/RNG 和已消费 token；只恢复模型权重会改变后续数据顺序，破坏精确复现。


In [ ]:
def draw_domains139(probs,n,seed):  # 定义本节可复用的核心函数。
    # searchsorted 显式完成 inverse-CDF 采样。
    r=np.random.default_rng(seed).random(n); return np.searchsorted(np.cumsum(probs),r,side="right")  # 计算并保存当前步骤的中间状态。
draws_a139=draw_domains139(p05_139,1000,7); draws_b139=draw_domains139(p05_139,1000,7)  # 计算并保存当前步骤的中间状态。
assert np.array_equal(draws_a139,draws_b139)  # 用受控断言验证关键不变量。
assert set(draws_a139)<=set(range(3))  # 用受控断言验证关键不变量。
assert abs(np.mean(draws_a139==0)-p05_139[0])<.06  # 用受控断言验证关键不变量。


## 5. DoReMi 类更新关注相对 reference 的域 excess loss

proxy 在某域比 reference 差得越多，该域权重经 exponentiated-gradient 增长；随后归一化。这里展示机制而非复现完整论文：真实训练还涉及 reference loss 平滑、Group DRO、proxy 规模和平均权重。绝对高 loss 可能只是噪声，不能不看 reference 就盲目加权。


In [ ]:
def dro_update139(weights,proxy_loss,ref_loss,eta):  # 定义本节可复用的核心函数。
    # excess loss 衡量相对基线欠拟合，而非域本身绝对难度。
    excess=np.asarray(proxy_loss)-np.asarray(ref_loss); w=np.asarray(weights)*np.exp(eta*excess); return w/w.sum(),excess  # 计算并保存当前步骤的中间状态。
w0_139=np.ones(3)/3; w1_139,excess139=dro_update139(w0_139,[2.0,1.5,1.1],[1.8,1.0,1.0],2.0)  # 计算并保存当前步骤的中间状态。
assert np.argmax(w1_139)==1  # 用受控断言验证关键不变量。
assert math.isclose(float(w1_139.sum()),1.0)  # 用受控断言验证关键不变量。
assert excess139[1]>excess139[0]>excess139[2]  # 用受控断言验证关键不变量。


## 6. 优化权重必须经过 floor、cap 与许可门禁

floor 防止基础语言/安全域消失，cap 防止噪声域因短期高 loss 独占预算。投影要保持总和为 1；不允许的域先移除。质量、重复 epoch、许可证和下游关键 slice 都可形成硬约束，不能被 loss 优化覆盖。


In [ ]:
def bounded_simplex139(raw,lo,hi):  # 定义本节可复用的核心函数。
    raw=np.asarray(raw,dtype=float); p=np.zeros_like(raw); free=set(range(len(raw))); mass=1.0  # 计算并保存当前步骤的中间状态。
    # 迭代固定触碰上下界的域，再按原始比例分剩余质量。
    while free:  # 在终止条件满足前持续推进状态。
        idx=sorted(free); proposal=mass*raw[idx]/raw[idx].sum()  # 计算并保存当前步骤的中间状态。
        # 每次只固定一个越界域，随后用新剩余质量重新计算其他域。
        if np.max(proposal)>hi:  # 按当前条件选择后续控制路径。
            i=idx[int(np.argmax(proposal))]; p[i]=hi; mass-=hi; free.remove(i); continue  # 计算并保存当前步骤的中间状态。
        if np.min(proposal)<lo:  # 按当前条件选择后续控制路径。
            i=idx[int(np.argmin(proposal))]; p[i]=lo; mass-=lo; free.remove(i); continue  # 计算并保存当前步骤的中间状态。
        p[idx]=proposal; break  # 计算并保存当前步骤的中间状态。
    return p  # 返回当前分支计算出的结果。
bounded139=bounded_simplex139([.95,.04,.01],.1,.7)  # 计算并保存当前步骤的中间状态。
assert np.allclose(bounded139,[.7,.2,.1])  # 用受控断言验证关键不变量。
assert math.isclose(float(bounded139.sum()),1.0)  # 用受控断言验证关键不变量。
assert np.all((bounded139>=.1)&(bounded139<=.7))  # 用受控断言验证关键不变量。


## 7. 跨域重复样本只能有一个训练归属

同一代码片段同时出现在 Web 和 code 域会被重复加权。全局 exact/near dedup 后为 document family 指定 canonical owner，并保留原始来源作审计；否则调 mixture 时既改变主题比例又暗中改变重复率。


In [ ]:
docs139=[("web","alpha"),("code","alpha"),("code","beta"),("math","gamma")]  # 计算并保存当前步骤的中间状态。
def canonical139(rows,priority):  # 定义本节可复用的核心函数。
    # 相同内容按预设域优先级归属，避免依赖输入顺序。
    grouped={}  # 计算并保存当前步骤的中间状态。
    for domain,text in rows: grouped.setdefault(hashlib.sha256(text.encode()).hexdigest(),[]).append((domain,text))  # 遍历输入元素以累积或检查结果。
    return [min(v,key=lambda x:priority[x[0]]) for v in grouped.values()]  # 返回当前分支计算出的结果。
unique139=canonical139(docs139,{"code":0,"math":1,"web":2})  # 计算并保存当前步骤的中间状态。
assert len(unique139)==3  # 用受控断言验证关键不变量。
assert ("code","alpha") in unique139  # 用受控断言验证关键不变量。
assert len({text for _,text in unique139})==3  # 用受控断言验证关键不变量。


## 8. 监控目标分布、实际分布与有效 epoch

数据 loader 故障、短文 padding、过滤和 worker 重试会让实际消费偏离计划。按有效非 padding token 统计 observed share、KL/最大偏差、每域有效 epoch、loss 与下游 slice；超阈值停止而不是训练结束后才发现 code 域没读到。


In [ ]:
observed139=np.bincount(draws_a139,minlength=3)/len(draws_a139)  # 计算并保存当前步骤的中间状态。
def kl139(p,q):  # 定义本节可复用的核心函数。
    # 加微小下界仅用于避免日志中的零概率数值错误。
    p=np.clip(p,1e-12,1); q=np.clip(q,1e-12,1); return float(np.sum(p*np.log(p/q)))  # 计算并保存当前步骤的中间状态。
epochs139=quota139/np.asarray(sizes139)  # 计算并保存当前步骤的中间状态。
assert kl139(observed139,p05_139)>=0  # 用受控断言验证关键不变量。
assert np.max(np.abs(observed139-p05_139))<.06  # 用受控断言验证关键不变量。
assert epochs139[-1]>epochs139[0]  # 用受控断言验证关键不变量。


## 面试总结

完整链路是：**版本化 domain/许可/唯一 token → `n^α` 温度平滑 → largest-remainder 生成精确 quota → sampler/RNG 可恢复 → proxy 相对 reference 的 excess loss 调权 → floor/cap/质量硬门禁 → 全局重复归属 → 按有效 token 监控实际比例和有效 epoch → 多域 held-out 与下游 slice 联合选型**。配比优化不能代替数据治理。

延伸阅读：[DoReMi](https://arxiv.org/abs/2305.10429)、[The Pile](https://arxiv.org/abs/2101.00027)、[DataComp-LM](https://arxiv.org/abs/2406.11794)。
